# EDA Dataset Analysis

**Dataset ID**: SEAMLESS.INTERNAL v1
**Telemetry**: EVTX-ATTACK-SAMPLES v1 (descriptive independent corpus)
**Environment**: Python 3.14 + NetworkX 3.6.1

Objective: Conduct an Exploratory Data Analysis on the AD Graph to establish a structural baseline, audit data quality (#56), quantify collector bias (#57), and align audit evidence boundaries with #58.


## Environment Setup & Imports


In [ ]:
import sys
from pathlib import Path
import json
import networkx as nx
from collections import Counter
import pprint

# Set path relative to repo root
try:
    ROOT = Path(__file__).parents[1]
except NameError:
    ROOT = Path.cwd().resolve()
    if ROOT.name == "notebooks":
        ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.snapshot_manifest import load_manifest, verify_manifest
from scripts.eda_dataset_analysis import (
    build_networkx_graph,
    compute_connectivity_metrics,
    compute_reachability_metrics,
    compute_privilege_concentration,
    compute_relationship_mixing,
    compute_group_nesting_pathology,
    compute_path_redundancy,
    compute_structural_outliers,
    parse_v3_dataset,
    parse_v4_dataset,
    parse_telemetry,
    audit_data_quality,
    compare_collector_bias,
)
print("Environment initialized successfully.")


## Data Ingestion & Graph Instantiation
Ingest raw BloodHound v3 dataset for SEAMLESS.INTERNAL and construct NetworkX graphs.


In [ ]:
seamless_path = ROOT / "data" / "raw" / "SEAMLESS"
seamless_manifest_path = ROOT / "data" / "samples" / "manifests" / "seamless-raw-v1.json"
seamless_manifest = load_manifest(seamless_manifest_path)
seamless_issues = verify_manifest(seamless_path, seamless_manifest)
if seamless_issues:
    raise ValueError(f"SEAMLESS manifest verification failed: {seamless_issues}")
G_multi, G_di, nodes = build_networkx_graph(seamless_path, manifest=seamless_manifest)
print(f"EDA graph instantiated: {G_multi.number_of_nodes()} nodes, {G_multi.number_of_edges()} raw/extracted entries (not canonical edges)")
# Distinguish: raw input records (3,369), unique snapshot OIDs (3,368), and materialized graph nodes (3,370)
print(f"Materialized graph nodes: {len(G_di.nodes())} (3,368 internal unique + 2 external well-known)")
print(f"Reconciled snapshot nodes: {sum(1 for n, d in nodes.items() if not d.get(\"is_external\"))}")


## Data Quality & Missingness Audit (Issue #56)
Verify record counts, dropped records, duplicate ObjectIdentifiers, referential integrity, and 7-class missingness taxonomy.


In [ ]:
data_quality = audit_data_quality(seamless_path, manifest=seamless_manifest)
print('Data Quality Audit Summary:')
print(f"  Records parsed: {data_quality['valid_parsed']} (Dropped: {data_quality['dropped_records']}, Malformed: {data_quality['malformed_records']})")
print(f"  Duplicate OIDs: {data_quality['duplicate_detection']['duplicate_ObjectIdentifiers']}")
print(f"  Referential integrity: {data_quality['referential_integrity']['integrity_percentage']}% ({data_quality['referential_integrity']['internal_references']} internal, {data_quality['referential_integrity']['referenced_well_known_external']} external, {data_quality['referential_integrity']['dangling_references']} dangling)")
print('Missingness Taxonomy Breakdown:')
for attr, info in data_quality['missingness_taxonomy'].items():
    print(f"  - {attr}: {info}")
print('Missingness Concentration Breakdown:')
for k, v in data_quality['missingness_concentration'].items():
    print(f"  - {k}: {v}")
print('Positive raw/evidence observations (not canonical edge counts):')
for ek, info in data_quality['unknown_rate_by_edge_type'].items():
    positive = info.get('positive_raw_evidence_rate', info.get('positive_evidence_rate', info.get('positive_observation_rate')))
    positive_text = f'{positive}%' if positive is not None else 'n/a'
    field_rate = info.get('field_presence_rate')
    field_text = f'{field_rate}%' if field_rate is not None else 'n/a'
    print(f"  - {ek}: positive_raw_evidence={positive_text}, field_presence={field_text}, completeness={info['collection_completeness_status']} ({info['status']})")
print('Collector Error / Access Denied Telemetry:')
print(f"  - Status: {data_quality['collector_error_and_access_denied']['status']}")
print(f"  - Limitation: {data_quality['collector_error_and_access_denied']['limitation_note']}")


## Topological Connectivity
Weakly Connected Components (WCC), Strongly Connected Components (SCC), clustering coefficient, giant component diameter, and node topological roles.


In [ ]:
connectivity = compute_connectivity_metrics(G_di)
print('Connectivity Metrics:')
pprint.pprint(connectivity)


## Tier-0 Reachability Analysis
Reachability to Domain Admins and Tier-0 targets, evaluated against non-target security principals (User, Group, Computer) and total graph nodes, along with shortest-hop distribution.


In [ ]:
reachability = compute_reachability_metrics(G_di, nodes)
print('Tier-0 Reachability:')
print(f"  Reachable non-target principals: {reachability['reachable_count']} ({reachability['reachable_non_target_principal_percentage']}%)")
print(f"  Reachable total graph nodes: {reachability['reachable_count']} ({reachability['reachable_graph_nodes_percentage']}%)")
print('By Class:')
pprint.pprint(reachability['by_class'])
print('Shortest-Hop Distribution:')
pprint.pprint(reachability['shortest_hop_distribution'])


## Privilege Concentration & Inequality
Gini coefficients (out, in, total degree), top 1% and 5% edge concentration, top privilege hubs, and heavy-tailed distribution characterization.


In [ ]:
privilege = compute_privilege_concentration(G_multi, nodes)
print('Privilege Concentration:')
print(f"  Distribution: {privilege.get('degree_distribution')}")
print(f"  Description: {privilege.get('description')}")
print('Gini coefficients:')
pprint.pprint(privilege['gini_coefficients'])
print('Top shares:')
pprint.pprint(privilege['top_shares'])
print('Top 5 hubs:')
pprint.pprint(privilege['top_10_hubs'][:5])


## Relationship Mixing Matrix
Display top `source_kind x edge_kind x target_kind` triplets with counts and percentages.


In [ ]:
mixing = compute_relationship_mixing(G_multi, nodes)
print('Relationship Mixing (Top 15):')
pprint.pprint(mixing[:15])


## Group Nesting Pathology
DAG acyclicity verification, branching factor statistics, and nesting depth distribution.


In [ ]:
group_nodes = {nid for nid, attr in nodes.items() if attr.get('kind') == 'Group'}
group_subgraph = G_di.subgraph(group_nodes).copy()
nesting = compute_group_nesting_pathology(group_subgraph, nodes)
print('Group Nesting Pathology:')
pprint.pprint(nesting)


## Path Redundancy & Choke Point Analysis
Multi-path diversity vs edge/node-disjoint paths on representative samples, with search capping indicator (`capped_at: 20`, `truncated: True/False`).


In [ ]:
redundancy = compute_path_redundancy(G_di, nodes)
print('Path Redundancy:')
pprint.pprint(redundancy)


## Structural Outliers
Top out-degree and in-degree outliers (> p99).

**Note: Structural topological anomalies only; does not imply malicious activity.**


In [ ]:
outliers = compute_structural_outliers(G_multi, nodes)
print('Structural Outliers:')
pprint.pprint(outliers)


## Independent telemetry corpus inventory (Issue #79 Phase B1)
EventID frequencies and upstream `EVTX_Tactic` labels describe the independent corpus only.
The runtime semantic catalog remains `configs/attack_semantics.yml`; these counts do not establish relationships or attacks.


In [ ]:
telemetry_dir = ROOT / "data" / "raw" / "telemetry"
telemetry_manifest_path = ROOT / "data" / "samples" / "manifests" / "telemetry-raw-v1.json"
telemetry_manifest = load_manifest(telemetry_manifest_path)
telemetry_issues = verify_manifest(telemetry_dir, telemetry_manifest)
if telemetry_issues:
    raise ValueError(f"Telemetry manifest verification failed: {telemetry_issues}")
telemetry_path = telemetry_dir / "evtx_data.csv"
telemetry_data = parse_telemetry(telemetry_path, manifest=telemetry_manifest)
print("Boundary note:", telemetry_data["telemetry_evidence_inventory"]["boundary_note"])
print("Upstream tactic annotations:")
pprint.pprint(telemetry_data["upstream_tactic_annotations"])
print("Top corpus EventIDs:")
pprint.pprint(telemetry_data["top_event_ids"])


## Cross-Dataset Comparison & Collector Bias Analysis (Issue #57)
Comparing node counts across datasets and quantifying collector bias between SharpHound C# and BloodHound.py on GOADv2 NORTH.


In [ ]:
seamless_data = parse_v3_dataset(ROOT / 'data' / 'raw' / 'SEAMLESS')
north_data = parse_v4_dataset(ROOT / 'data' / 'raw' / 'NORTH')
ad_sample_data = parse_v4_dataset(ROOT / 'data' / 'raw' / 'ad_sampledata')

print('SEAMLESS nodes:', seamless_data.get('total_nodes'))
print('NORTH nodes:', north_data.get('total_nodes'))
print('ad_sampledata nodes:', ad_sample_data.get('total_nodes'))

print('\nNormalized Comparisons:')
for name, d in [('SEAMLESS', seamless_data), ('NORTH', north_data), ('ad_sampledata', ad_sample_data)]:
    nr = d.get('normalized_ratios', {})
    print(f"  {name}: density={d.get('graph_density'):.6f}, priv_ratio={nr.get('privileged_principal_ratio')}, acl_density={nr.get('acl_density')}, session_density={nr.get('session_density')} ({nr.get('session_density_status')})")

north_dir = ROOT / 'data' / 'raw' / 'NORTH'
python_dir = ROOT / 'data' / 'raw' / 'NORTH_CE_PYTHON'
if north_dir.exists() and python_dir.exists():
    collector_bias = compare_collector_bias(north_dir, python_dir)
    print('\nCollector Bias Comparison (NORTH SharpHound vs BloodHound.py):')
    print(f"  SharpHound edges: {collector_bias['Total_SharpHound_edges']}")
    print(f"  BloodHound.py edges: {collector_bias['Total_BloodHound.py_edges']}")
    print(f"  Intersection edges: {collector_bias['Common_edges']}")
    print(f"  Union edges: {collector_bias['Union_edges']}")
    print(f"  Overall Jaccard Similarity: {collector_bias['Overall_Jaccard_Similarity']}")
    print('  Per-edge breakdown sample (MemberOf, GenericAll, WriteDacl):')
    for ek in ['MemberOf', 'GenericAll', 'WriteDacl']:
        if ek in collector_bias['Per_edge_kind_breakdown']:
            print(f"    {ek}: {collector_bias['Per_edge_kind_breakdown'][ek]}")


## Final Summary Section

### Q&A
- **Why does the static AD graph require telemetry?** Because static directory permissions do not equal active exploitation; telemetry provides independent descriptive audit evidence of host/network activity.
- **Why is the giant SCC 1,172 nodes?** It represents the dense core of bidirectional reachability and administrative privilege cycles.
- **Why does choke point analysis matter?** Identifying choke points reveals critical convergence nodes where defensive controls and monitoring break lateral movement paths.

### Insights and Next Steps
- Maintain strict boundary separation between structural topology (Layer 1) and attack semantics (Layer 2) per Issue #58.
- Do not extrapolate candidate telemetry events directly into execution weights until semantic preconditions and transition rules are finalized.


In [ ]:
print('=== DATA ANALYSIS KEY FINDINGS ===')
print(f"- Data Quality (#56): {data_quality['valid_parsed']} parsed records ({data_quality['dropped_records']} duplicate dropped) yielding {data_quality['unique_nodes_materialized']} unique snapshot nodes; EDA reference subset {data_quality['referential_integrity']['integrity_percentage']}% resolved or boundary ({data_quality['referential_integrity']['internal_references']} internal references, {data_quality['referential_integrity']['referenced_well_known_external']} external references); broader #94 audit has unresolved references.")
print(f"- Missingness & Quality (#56): OS field unpopulated on {data_quality['missingness_taxonomy']['operatingSystem']['percentage_missing']}% of recorded hosts; HasLAPS collector-observed false {data_quality['missingness_taxonomy']['haslaps']['percentage_observed_false']}%; collector error/access-denied is {data_quality['collector_error_and_access_denied']['status']}; collection completeness remains unknown.")
print(f"- Collector Bias (#57): SharpHound ({collector_bias['Total_SharpHound_edges']} edges) vs BloodHound.py ({collector_bias['Total_BloodHound.py_edges']} edges) yields overall Jaccard of {collector_bias['Overall_Jaccard_Similarity']:.4f} across {collector_bias['Union_edges']} union edges.")
print(f"- EDA raw reachability (#55): {reachability['reachable_count']} non-target principal records ({reachability['reachable_non_target_principal_percentage']:.2f}%) have graph paths to Tier-0-labeled groups (mean shortest raw path {reachability['shortest_hop_distribution']['mean']} hops); effective entitlement and attack applicability are unverified.")
print(f"- Degree & Topology (#55): Out-degree Gini {privilege['gini_coefficients']['out_degree']:.4f}, graph density {seamless_data['graph_density']:.6f}, edge-to-node ratio {seamless_data['edge_to_node_ratio']:.2f}.")
